# Phase 03: Base Model Evaluation - Code Mentor AI

In this notebook, we establish our **baseline performance** before fine-tuning. We evaluate the pretrained **`Qwen/Qwen2.5-0.5B-Instruct`** model on a fixed set of test cases covering code generation, explanation, and debugging.

### Notebook Outline:
1. Project Objective
2. Install and Import Libraries
3. Set Random Seed and Device
4. Load Tokenizer
5. Load Base Model
6. Prepare Evaluation Test Cases
7. Create Text Generation Function
8. Generate Base Model Responses
9. Inspect Sample Predictions
10. Evaluate Basic Performance
11. Save Baseline Results to CSV
12. Final Observations

# 02. Install and Import Libraries

In [1]:
!pip install transformers torch accelerate

In [2]:
# Uncomment if running in a fresh environment: !pip install transformers torch accelerate pandas
import torch
import time
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Libraries imported successfully!")
print(f"PyTorch Version: {torch.__version__}")

Libraries imported successfully!
PyTorch Version: 2.11.0+cu128


# 03. Set Random seed and Device

In [3]:
import random 
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)
device='cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device.upper()}')

Using device: CUDA


# 04. Load Tokenizer

In [4]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
print(f"Loading tokenizer for {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
# Ensure padding token is set properly
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Tokenizer loaded successfully!')

Loading tokenizer for Qwen/Qwen2.5-0.5B-Instruct...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded successfully!


# 05. Load Base Model

In [5]:
print(f"Loading base model {model_id}...")

# Load in half-precision (float16) for efficient inference on Kaggle GPU
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)

print("Base model loaded successfully!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading base model Qwen/Qwen2.5-0.5B-Instruct...


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded successfully!


# 06. Prepare Evaluation Test Cases

In [6]:
# Fixed test suite covering generation, explanation, and debugging
eval_test_cases = [
    {
        "task_id": "gen_01",
        "category": "Code Generation",
        "prompt": "Write a Python function using two pointers to check if a given string is a palindrome, ignoring non-alphanumeric characters."
    },
    {
        "task_id": "gen_02",
        "category": "Code Generation",
        "prompt": "Implement a Python function to compute the Nth Fibonacci number using dynamic programming with O(1) space complexity."
    },
    {
        "task_id": "exp_01",
        "category": "Code Explanation",
        "prompt": "Explain how Python's list comprehension works under the hood and why it is generally faster than a standard for-loop."
    },
    {
        "task_id": "exp_02",
        "category": "Code Explanation",
        "prompt": "What is the difference between shallow copy and deep copy in Python? Provide a quick code example."
    },
    {
        "task_id": "bug_01",
        "category": "Debugging",
        "prompt": "Debug this Python snippet that modifies a list while iterating:\n```python\nnums = [1, 2, 3, 4, 4, 5]\nfor n in nums:\n    if n == 4:\n        nums.remove(n)\nprint(nums)\n```\nExplain why it fails and provide the corrected code."
    }
]

print(f"Prepared {len(eval_test_cases)} evaluation test cases across code generation, explanation, and debugging.")

Prepared 5 evaluation test cases across code generation, explanation, and debugging.


# 07. Create Text Generation Function

In [7]:
def generate_response(prompt_text, max_new_tokens=512):
    system_prompt = (
        "You are Code Mentor AI, an expert software engineer and programming tutor. "
        "Your goal is to help users write clean code, explain complex algorithms step-by-step, "
        "and debug errors thoroughly."
    )
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt_text}
    ]
    
    # Apply Qwen's chat template
    text_input = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer([text_input], return_tensors="pt").to(device)
    
    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,  # Low temperature for stable evaluation
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )
    end_time = time.time()
    
    generation_time = end_time - start_time
    
    # Decode only the generated response (excluding the input prompt)
    input_len = inputs['input_ids'].shape[1]
    generated_tokens = outputs[0][input_len:]
    response_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    
    token_count = len(generated_tokens)
    
    return {
        "response": response_text,
        "generation_time_sec": round(generation_time, 2),
        "token_count": token_count,
        "tokens_per_sec": round(token_count / generation_time, 2) if generation_time > 0 else 0
    }

print("Generation function defined successfully!")

Generation function defined successfully!


# 08. Generate Base Model Responses

In [8]:
results = []

print("Running baseline evaluation on base model...\n")
for idx, case in enumerate(eval_test_cases, 1):
    print(f"Evaluating [{case['task_id']}] - {case['category']}...")
    res = generate_response(case['prompt'])
    
    results.append({
        "task_id": case['task_id'],
        "category": case['category'],
        "prompt": case['prompt'],
        "base_response": res['response'],
        "generation_time_sec": res['generation_time_sec'],
        "token_count": res['token_count'],
        "tokens_per_sec": res['tokens_per_sec']
    })

print("\nBaseline generation completed for all test cases!")

Running baseline evaluation on base model...

Evaluating [gen_01] - Code Generation...
Evaluating [gen_02] - Code Generation...
Evaluating [exp_01] - Code Explanation...
Evaluating [exp_02] - Code Explanation...
Evaluating [bug_01] - Debugging...

Baseline generation completed for all test cases!


# 09. Inspect Sample Predictions

In [9]:
# Let's inspect the first evaluation result
print("=" * 60)
print(f"Sample Inspection: Task ID -> {results[0]['task_id']} ({results[0]['category']})")
print("=" * 60)
print(f"PROMPT:\n{results[0]['prompt']}\n")
print(f"BASE MODEL RESPONSE:\n{results[0]['base_response']}")
print("=" * 60)

Sample Inspection: Task ID -> gen_01 (Code Generation)
PROMPT:
Write a Python function using two pointers to check if a given string is a palindrome, ignoring non-alphanumeric characters.

BASE MODEL RESPONSE:
To create a Python function that checks if a given string is a palindrome, ignoring non-alphanumeric characters, you can use the following approach:

1. Convert the input string to lowercase.
2. Use two pointers to traverse the string from both ends towards the center.
3. Compare characters at these positions: left and right (or vice versa).
4. If all corresponding characters match, return `True`; otherwise, return `False`.

Here's how you could implement this:

```python
def is_palindrome(s):
    """
    Check if the given string is a palindrome, ignoring non-alphanumeric characters.

    :param s: The string to check.
    :return: True if the string is a palindrome, False otherwise.
    """
    # Remove non-alphanumeric characters and convert to lowercase
    cleaned_s = ''.joi

# 10. Evaluate Basic Performance

In [10]:
df_results = pd.DataFrame(results)

print("=" * 60)
print("BASELINE PERFORMANCE METRICS SUMMARY")
print("=" * 60)
print(f"Average Generation Time: {df_results['generation_time_sec'].mean():.2f} seconds")
print(f"Average Token Count: {df_results['token_count'].mean():.1f} tokens")
print(f"Average Speed: {df_results['tokens_per_sec'].mean():.2f} tokens/sec")
print("=" * 60)
df_results[['task_id', 'category', 'generation_time_sec', 'token_count', 'tokens_per_sec']]

BASELINE PERFORMANCE METRICS SUMMARY
Average Generation Time: 21.97 seconds
Average Token Count: 476.6 tokens
Average Speed: 21.75 tokens/sec


,task_id,category,generation_time_sec,token_count,tokens_per_sec
0,gen_01,Code Generation,25.00,512,20.48
1,gen_02,Code Generation,21.72,473,21.78
2,exp_01,Code Explanation,23.17,512,22.10
3,exp_02,Code Explanation,23.19,512,22.08
4,bug_01,Debugging,16.78,374,22.29


# 11. Save Baseline Results to CSV (Kaggle Optimized)

In [11]:
import os

output_dir = "/kaggle/working"
csv_path = os.path.join(output_dir, "base_model_baseline_results.csv")

df_results.to_csv(csv_path, index=False)
print(f"Baseline results successfully saved to: {csv_path}")

Baseline results successfully saved to: /kaggle/working/base_model_baseline_results.csv


# Conclusion

### 12. Final Observations
- The base `Qwen2.5-0.5B-Instruct` model successfully runs inference and generates structured code explanations.
- However, prior to fine-tuning, responses might rely on general instruction-following rather than a specialized **Code Mentor AI** persona, or they might lack deep debugging depth on complex edge cases.
- These results are safely stored in `base_model_baseline_results.csv`, ready to be compared side-by-side with our fine-tuned model's evaluation metrics in Phase 05!